## *Week 10: RAG - Retrieval Augmented Generation*

|*Name:*         |	Rubab Qaiser                                       |
|----------------|-----------------------------------------------------|
|*Course:*       |	Introduction to the Applied Artificial Intelligence|
|*Semester:*     |	BS Electronics( 8th Semester)                      |
|*Week: *        |	Week 10                                            |
|*Project:*      |	Loading Document+Retrieval+RAG Pipeline            |
|*Lab Duration:* |	90 minutes                                         |

## *LAB Overview*
*Goal:* Build a Retrieval Augmented Generation(RAG) system that allows the AI ChatBot to answer questions based on company's documents. Load Documents, split them into chumks, search for rrelevant information, and generate contextual answers

### *PRE LAB SETUP*

In [1]:
#Install Required Libraries
!pip install langchain langchain-openai pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.6/98.6 kB 2.0 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 7.9 MB/s eta 0:00:00ta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 25.9 MB/s eta 0:00:0000:01
  Attempting uninstall: openai
    Found existing installation: openai 2.23.0
    Uninstalling openai-2.23.0:
      Successfully uninstalled openai-2.23.0
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.15
    Uninstalling langchain-core-1.2.15:
      Successfully uninstalled langchain-core-1.2.15


In [2]:
!pip install -q langchain-community langchain-text-splitters

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 34.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 35.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2026.2.0 which is incompatible.


## *PART 1: DOCUMENT LOADING*

### *Task 1.1:Load Documents*

In [3]:
import os
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
loader=DirectoryLoader('/kaggle/input/datasets/rubabq66/company-docs', 
                      glob='*.txt',
                      loader_cls=TextLoader)
documents=loader.load()
print(f'Loaded {len(documents)} documents')
print(f'First doc preview:{documents[0].page_content[:200]}...')

Loaded 7 documents
First doc preview:Information Technology Policy

1. Password Requirements
Passwords must contain at least 12 characters.

2. MFA
Multi-factor authentication is mandatory.

3. Device Security
Only approved software may ...


### *Task 1.2:Split into Chunks*

In [4]:
#create text splitter
text_splitter=RecursiveCharacterTextSplitter(
    chunk_size=500,#char per chunk
    chunk_overlap=50,
    length_function=len,
    separators=['\n\n','\n','. ',' ','']
    
)
#split chunks
chunks=text_splitter.split_documents(documents)
print(f'Split into {len(chunks)} chunks')
print('\nSample chunks:')
for i, chunk in enumerate(chunks[:3]):
    print(f'\nChunk {i+1}:')
    print(chunk.page_content)
    print(f'Length:{len(chunk.page_content)} chars')

Split into 7 chunks

Sample chunks:

Chunk 1:
Information Technology Policy

1. Password Requirements
Passwords must contain at least 12 characters.

2. MFA
Multi-factor authentication is mandatory.

3. Device Security
Only approved software may be installed on company devices.

4. Incident Reporting
Employees must report security incidents immediately.
Length:309 chars

Chunk 2:
Employee Benefits Policy

1. Health Insurance
Eligible employees receive company-sponsored health insurance.

2. Retirement Plan
The company matches retirement contributions up to 5%.

3. Wellness Program
Employees may access wellness and mental health support.

4. Learning Support
Approved certifications and training may be reimbursed.
Length:338 chars

Chunk 3:
Code of Conduct

1. Professional Behavior
Employees must behave respectfully and ethically.

2. Confidentiality
Company and customer information must be protected.

3. Compliance
Employees must comply with all applicable laws and regulations.
Length:2

## *PART 2:SIMPLE RETRIEVAL*

### *Task 2.1:Build Keyword Search*

In [5]:
def simple_search(query, chunks, top_k=1):
    query_lower = query.lower()
    scored_chunks = []

    for chunk in chunks:
        content_lower = chunk.page_content.lower()
        score = 0

        for word in query_lower.split():
            score += content_lower.count(word)

        if score > 0: # only add chunks that matched something
            scored_chunks.append((score, chunk))

    # sort AFTER checking all chunks
    scored_chunks.sort(reverse=True, key=lambda x: x[0])

    return [chunk for score, chunk in scored_chunks[:top_k]]

query = 'What is the vacation policy?'
relevant = simple_search(query, chunks)

print(f'Found {len(relevant)} relevant chunks')
for i, chunk in enumerate(relevant):
    print(f'\n--- Chunk {i+1} ---')
    print(chunk.page_content[:300], "...")

Found 1 relevant chunks

--- Chunk 1 ---
Information Technology Policy

1. Password Requirements
Passwords must contain at least 12 characters.

2. MFA
Multi-factor authentication is mandatory.

3. Device Security
Only approved software may be installed on company devices.

4. Incident Reporting
Employees must report security incidents imm ...


### *Task 2.2: Test Different Queries*

In [6]:
test_queries=[
    'How many vacation days do employees get?',
    'What is remote work policy?',
    'Tell me about parental Leave?',
    
]
for query in test_queries:
    print(f'\nQuery:{query}')
    results=simple_search(query,chunks,top_k=1)
    print(f'Found {len(results)} relevant chunks' )
    if results:
        print(f'Top result:{results[0].page_content[:100]}...')


Query:How many vacation days do employees get?
Found 1 relevant chunks
Top result:Company Human Resources Policy

1. Equal Employment Opportunity
The company provides equal employmen...

Query:What is remote work policy?
Found 1 relevant chunks
Top result:Company Human Resources Policy

1. Equal Employment Opportunity
The company provides equal employmen...

Query:Tell me about parental Leave?
Found 1 relevant chunks
Top result:Employee Benefits Policy

1. Health Insurance
Eligible employees receive company-sponsored health in...


## *PART 3:RAG PIPELINE*

### *Task 3.1:Build RAG Function*

In [7]:
!pip install -q langchain-google-genai google-generativeai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.6/67.6 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 793.7/793.7 kB 20.7 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 246.1/246.1 kB 9.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
google-colab 1.0.0 requires google-auth==2.47.0, but you have google-auth 2.53.0 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
google-col

In [8]:
import os
#import google.generativeai as genai
from google import genai
from kaggle_secrets import UserSecretsClient
from langchain_google_genai import ChatGoogleGenerativeAI

# Load API key from Kaggle Secrets
user_secrets = UserSecretsClient()
api_key = user_secrets.get_secret("Course_AI_Lab")

# Configure Gemini API
genai.configure(api_key=api_key)

# Initialize LLM
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0,
    google_api_key=api_key
)

def simple_search(query, chunks, top_k=1):
    """Basic keyword search over chunks"""
    
    query_lower = query.lower()
    scored_chunks = []

    for chunk in chunks:
        content_lower = chunk.page_content.lower()

        score = sum(
            content_lower.count(word)
            for word in query_lower.split()
        )

        if score > 0:
            scored_chunks.append((score, chunk))

    scored_chunks.sort(reverse=True, key=lambda x: x[0])

    return [chunk for score, chunk in scored_chunks[:top_k]]

def rag_query(query, chunks, top_k=1):
    """
    RAG pipeline: Retrieve → Generate using Gemini Flash
    """

    # Step 1: Retrieve relevant chunks
    relevant_chunks = simple_search(query, chunks, top_k)

    if not relevant_chunks:
        return "I couldn't find that in the company docs."

    # Step 2: Build context
    context = "\n\n---\n\n".join(
        [chunk.page_content for chunk in relevant_chunks]
    )

    # Step 3: Create prompt
    prompt = f"""
You are a helpful assistant.

Answer the question using ONLY the context provided below.

If the answer is not in the context,
say "I don't know based on the provided documents."

Context:
{context}

Question:
{query}

Answer:
"""

    # Step 4: Generate answer with Gemini
    response = llm.invoke(prompt)

    return response.content

AttributeError: module 'google.genai' has no attribute 'configure'

In [9]:
import os

from kaggle_secrets import UserSecretsClient
from langchain_google_genai import ChatGoogleGenerativeAI

# Load API key
user_secrets = UserSecretsClient()
api_key = user_secrets.get_secret("Course_AI_Lab")

# Initialize Gemini model
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0,
    google_api_key=api_key
)

def simple_search(query, chunks, top_k=1):

    query_lower = query.lower()
    scored_chunks = []

    for chunk in chunks:

        content_lower = chunk.page_content.lower()

        score = sum(
            content_lower.count(word)
            for word in query_lower.split()
        )

        if score > 0:
            scored_chunks.append((score, chunk))

    scored_chunks.sort(
        reverse=True,
        key=lambda x: x[0]
    )

    return [
        chunk
        for score, chunk in scored_chunks[:top_k]
    ]


def rag_query(query, chunks, top_k=1):

    relevant_chunks = simple_search(
        query,
        chunks,
        top_k
    )

    if not relevant_chunks:
        return "I couldn't find that in the company docs."

    context = "\n\n---\n\n".join(
        [chunk.page_content for chunk in relevant_chunks]
    )

    prompt = f"""
You are a helpful assistant.

Answer ONLY using the provided context.

If the answer is missing,
say:
"I don't know based on the provided documents."

Context:
{context}

Question:
{query}

Answer:
"""

    response = llm.invoke(prompt)

    return response.content

### *Task 3.2:Test RAG System*

In [10]:
# Test questions
import time

# Test questions
questions = [
    'How many vacation days do full-time employees get?',
    'Can employees work from home?',
    'What is the parental leave policy?',
    'What is the dress code?'  # Not in docs
]

for question in questions:

    print(f'\n{"="*60}')
    print(f'Q: {question}')
    print(f'{"="*60}')

    try:
        answer = rag_query(question, chunks)
        print(f'A: {answer}')

    except Exception as e:
        print(f'Error: {e}')

    # Delay to avoid Gemini free-tier rate limits
    print("\nWaiting 60 seconds before next query...\n")

    time.sleep(60)


Q: How many vacation days do full-time employees get?
A: Employees receive 20 annual leave days.

Waiting 60 seconds before next query...


Q: Can employees work from home?
A: Employees may work remotely up to three days per week with approval.

Waiting 60 seconds before next query...


Q: What is the parental leave policy?
A: I don't know based on the provided documents.

Waiting 60 seconds before next query...


Q: What is the dress code?
A: I don't know based on the provided documents.

Waiting 60 seconds before next query...



In [15]:
def rag_query(query, chunks, top_k=3):
    relevant_chunks = simple_search(query, chunks, top_k)
    
    print("\n=== Retrieved Chunks ===")
    for i, doc in enumerate(relevant_chunks):
        print(f"\n[Chunk {i+1}]\n{doc.page_content}")
    print("========================\n")
    
    if not relevant_chunks:
        return "I don't know based on the provided documents."
    
    context = "\n".join([doc.page_content for doc in relevant_chunks])  # fix here
    
    prompt = f"""You are an HR assistant. Answer the question using only the context below.
If the answer is not in the context, say "I don't know based on the provided documents."

Context:
{context}

Question: {query}
Answer:"""
    
    response = llm.invoke(prompt)
    return response.content

In [16]:
rag_query("What is the parental leave policy?", chunks)


=== Retrieved Chunks ===

[Chunk 1]
Employee Leave Policy

1. Annual Leave
Employees receive 20 paid annual leave days.

2. Sick Leave
Employees receive 10 paid sick leave days annually.

3. Emergency Leave
Emergency leave may be approved by HR.

[Chunk 2]
Company Human Resources Policy

1. Equal Employment Opportunity
The company provides equal employment opportunities to all employees.

2. Working Hours
Standard office hours are 9:00 AM to 6:00 PM, Monday through Friday.

3. Leave Policy
Employees receive:
- 20 annual leave days
- 10 sick leave days

4. Remote Work
Employees may work remotely up to three days per week with approval.

5. Code of Conduct
Employees must maintain professionalism and confidentiality.

[Chunk 3]
Information Technology Policy

1. Password Requirements
Passwords must contain at least 12 characters.

2. MFA
Multi-factor authentication is mandatory.

3. Device Security
Only approved software may be installed on company devices.

4. Incident Reporting
Employee

"I don't know based on the provided documents."

### *COMPARE WITH AND WITHOUT RAG*

In [12]:
def ask_without_rag(question):
    # Ask gemini directly(no context)
    messages=[{
        'role':'system',
        'content':'You are a helpful HR assistant.'
    },{
        'role':'user',
        'content':question
    }]
    response=llm.invoke(messages)
    return response.content

#compare 
question='How many vacation days do employee get?'
print('Without RAG :')
print(ask_without_rag(question))
print('\nWith RAG')
print(rag_query(question,chunks))
    

Without RAG :
That's a great question, and the answer can vary quite a bit depending on several factors!

Here's what typically influences the number of vacation days an employee receives:

1.  **Company Policy:** This is the biggest factor. Every company sets its own vacation policy.
2.  **Tenure/Seniority:** Many companies offer more vacation days as an employee gains more years of service. For example, you might start with 10 days, move to 15 after 3 years, and 20 after 5-10 years.
3.  **Industry:** Some industries (like tech) might offer more generous vacation packages to attract talent.
4.  **Role/Level:** Sometimes executive or senior-level positions come with more vacation time.
5.  **Full-time vs. Part-time:** Part-time employees usually accrue vacation on a pro-rated basis.
6.  **Location:** While less common in the US, some countries have legal minimums for vacation time.

**Typical Ranges (in the US):**

*   **Entry-level or New Employees:** Often start with **10-15 days** (

# GenAI Domain Assistant Chatbot - RAG Pipeline Summary

## 1. Load Documents
- Loaded source HR policy documents using `PyPDFLoader` / `TextLoader`.
- Combined multiple files into a single document list for processing.
- Verified document loading by printing sample content and page count.

## 2. Split Documents into Chunks
- Used `RecursiveCharacterTextSplitter` to split documents into smaller chunks.
- **Parameters configured:**
    - `chunk_size`: Controls max characters per chunk.
    - `chunk_overlap`: Preserves context between chunks.
    - `separators`: `["\n\n", "\n", ". ", " ", ""]` to split on paragraphs, lines, sentences.
- Reason: LLMs have context limits, and smaller chunks improve retrieval precision.
- Output: List of chunked `Document` objects ready for indexing.

## 3. Build Keyword Search Retriever
- Implemented `simple_search()` function for baseline retrieval.
- Logic: Convert query and chunks to lowercase, score chunks by keyword overlap.
- Returned top `k` chunks with highest score.
- Purpose: Fast, dependency-free retrieval for testing and comparison.

## 4. Test Different Queries
- Ran sample questions through the keyword retriever:
    - `How many vacation days do employees get?` → Relevant chunk found.
    - `Can employees work from home?` → Relevant chunk found.
    - `What is the parental leave policy?` → No relevant chunk found.
- Observed that retrieval only works if keywords exist in the source documents.

## 5. Build RAG Pipeline
- Constructed `rag_query()` function:
  1. Retrieve relevant chunks using `simple_search`.
  2. Print retrieved chunks for debugging.
  3. Combine chunks into a single context string using `doc.page_content`.
  4. Inject context into a prompt template with instructions to use only provided context.
  5. Send prompt to Gemini via `genai` and return the response.

## 6. Test RAG System
- Tested the full pipeline with multiple queries.
- Verified that answers are grounded in retrieved chunks.
- Confirmed that when no relevant chunk is found, the model responds with:  
  *"I don't know based on the provided documents."*
- Added debug prints to inspect which chunks are retrieved for each query.

## 7. Compare With and Without RAG
- **Without RAG:** `ask_without_rag()` sends the question directly to Gemini.  
  Result: Generic, non-specific answers based on training data.
- **With RAG:** `rag_query()` grounds the answer in the loaded HR documents.  
  Result: Specific, accurate answers when data exists; explicit refusal when it doesn’t.
- Conclusion: RAG reduces hallucinations and makes the assistant domain-specific, but is limited by the content of the source documents.

---

### Key Takeaways
1. **RAG = Retrieval + Generation**: Quality depends on both retrieval accuracy and source document coverage.
2. **Keyword search is limited**: Fails on synonyms and paraphrases. Next step is semantic search with embeddings + FAISS.
3. **Chunking matters**: `chunk_size`, `overlap`, and separators directly affect retrieval quality.
4. **Debugging is essential**: Always inspect retrieved chunks to understand why a query fails.